In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

print("Libraries loaded!")

Libraries loaded!


In [2]:
df = pd.read_csv('diabetic_data.csv')

# Repeat key cleaning steps
df = df.drop(columns=['max_glu_serum', 'A1Cresult', 'weight', 'payer_code', 'encounter_id', 'patient_nbr'])

for col in ['race', 'medical_specialty', 'diag_1', 'diag_2', 'diag_3']:
    df[col] = df[col].replace('?', 'Unknown')
df['race'] = df['race'].replace('?', df['race'].mode()[0])

df['readmitted'] = df['readmitted'].apply(lambda x: 1 if x == '<30' else 0)

categorical_cols = [col for col in df.select_dtypes(include=['object']).columns if col != 'readmitted']
df = pd.get_dummies(df, columns=categorical_cols, drop_first=True)

print(f"Data ready: {df.shape}")

C:\Users\DELL-PV\AppData\Local\Temp\ipykernel_4712\460152544.py:12: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_cols = [col for col in df.select_dtypes(include=['object']).columns if col != 'readmitted']


Data ready: (101766, 2403)


In [3]:
X = df.drop(columns=['readmitted'])
y = df['readmitted']

print(f"Features (X): {X.shape}")
print(f"Target (y): {y.shape}")

Features (X): (101766, 2402)
Target (y): (101766,)


In [4]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")

Training set: (81412, 2402)
Test set: (20354, 2402)


In [5]:
lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train, y_train)
print("Model trained!")

Model trained!


C:\Users\DELL-PV\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [6]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

lr_model = LogisticRegression(max_iter=1000)
lr_model.fit(X_train_scaled, y_train)
print("Model trained with scaled data!")

Model trained with scaled data!


In [7]:
y_pred = lr_model.predict(X_test_scaled)

print("=== LOGISTIC REGRESSION RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred)*100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred))

=== LOGISTIC REGRESSION RESULTS ===
Accuracy: 88.5%

Detailed Report:
              precision    recall  f1-score   support

           0       0.89      0.99      0.94     18069
           1       0.33      0.02      0.04      2285

    accuracy                           0.89     20354
   macro avg       0.61      0.51      0.49     20354
weighted avg       0.83      0.89      0.84     20354



In [8]:
lr_model_balanced = LogisticRegression(max_iter=1000, class_weight='balanced')
lr_model_balanced.fit(X_train_scaled, y_train)

y_pred_balanced = lr_model_balanced.predict(X_test_scaled)

print("=== LOGISTIC REGRESSION (BALANCED) RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_balanced)*100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred_balanced))

=== LOGISTIC REGRESSION (BALANCED) RESULTS ===
Accuracy: 63.8%

Detailed Report:
              precision    recall  f1-score   support

           0       0.92      0.65      0.76     18069
           1       0.16      0.54      0.25      2285

    accuracy                           0.64     20354
   macro avg       0.54      0.60      0.51     20354
weighted avg       0.83      0.64      0.70     20354



In [9]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(
    n_estimators=100,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)
rf_model.fit(X_train, y_train)
print("Random Forest trained!")

Random Forest trained!


In [10]:
y_pred_rf = rf_model.predict(X_test)

print("=== RANDOM FOREST RESULTS ===")
print(f"Accuracy: {accuracy_score(y_test, y_pred_rf)*100:.1f}%")
print("\nDetailed Report:")
print(classification_report(y_test, y_pred_rf))

=== RANDOM FOREST RESULTS ===
Accuracy: 88.8%

Detailed Report:
              precision    recall  f1-score   support

           0       0.89      1.00      0.94     18069
           1       0.55      0.00      0.01      2285

    accuracy                           0.89     20354
   macro avg       0.72      0.50      0.47     20354
weighted avg       0.85      0.89      0.84     20354



In [11]:
print("========= MODEL COMPARISON =========")
print(f"{'Model':<35} {'Accuracy':>10} {'Recall(1)':>10}")
print("-" * 57)

from sklearn.metrics import recall_score

lr_recall = recall_score(y_test, y_pred_balanced)
rf_recall = recall_score(y_test, y_pred_rf)
lr_acc = accuracy_score(y_test, y_pred_balanced)
rf_acc = accuracy_score(y_test, y_pred_rf)

print(f"{'Logistic Regression (balanced)':<35} {lr_acc*100:>9.1f}% {lr_recall:>10.2f}")
print(f"{'Random Forest (balanced)':<35} {rf_acc*100:>9.1f}% {rf_recall:>10.2f}")
print("=====================================")
print("\nWinner for healthcare use: higher recall wins")

========= MODEL COMPARISON =========
Model                                 Accuracy  Recall(1)
---------------------------------------------------------
Logistic Regression (balanced)           63.8%       0.54
Random Forest (balanced)                 88.8%       0.00

Winner for healthcare use: higher recall wins


In [12]:
import pickle

# Save the model and scaler
with open('model.pkl', 'wb') as f:
    pickle.dump(lr_model_balanced, f)

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

# Save the feature names
feature_names = X_train.columns.tolist()
with open('feature_names.pkl', 'wb') as f:
    pickle.dump(feature_names, f)

print("Model, scaler and features saved!")
print(f"Total features: {len(feature_names)}")

Model, scaler and features saved!
Total features: 2402
